# Step 3 - Evaluate Translation Quality (MedEV)

Evaluate GPT and Gemini outputs against MedEV English references.

Metrics:
- BLEU (higher is better)
- chrF++ (higher is better)
- TER (lower is better)

Outputs:
- `outputs_medev/results_overall.csv`
- `outputs_medev/results_by_length_bin.csv`

In [9]:
import os
import sys
import subprocess
import numpy as np
import pandas as pd

try:
    from sacrebleu.metrics import BLEU, CHRF, TER
except ModuleNotFoundError:
    print("Installing sacrebleu...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sacrebleu"])
    from sacrebleu.metrics import BLEU, CHRF, TER

OUT_DIR = "outputs_medev"
GPT_PATH = os.path.join(OUT_DIR, "translations_gpt.csv")
GEMINI_PATH = os.path.join(OUT_DIR, "translations_gemini.csv")

RESULT_OVERALL = os.path.join(OUT_DIR, "results_overall.csv")
RESULT_BY_BIN = os.path.join(OUT_DIR, "results_by_length_bin.csv")

# Speed knobs
N_BOOT = 120        # lower = faster (e.g., 80-150 for quick iteration)
RANDOM_SEED = 42

gpt_df = pd.read_csv(GPT_PATH)
gemini_df = pd.read_csv(GEMINI_PATH)

# Use only successfully translated rows for fair/fast evaluation
gpt_eval = gpt_df[~gpt_df["translated_en"].astype(str).str.startswith("ERROR", na=False)].copy()
gemini_eval = gemini_df[~gemini_df["translated_en"].astype(str).str.startswith("ERROR", na=False)].copy()

print(f"GPT rows total: {len(gpt_df):,} | eval rows: {len(gpt_eval):,}")
print(f"Gemini rows total: {len(gemini_df):,} | eval rows: {len(gemini_eval):,}")

GPT rows total: 8,959 | eval rows: 8,959
Gemini rows total: 8,959 | eval rows: 8,959


In [10]:
bleu = BLEU()
chrf = CHRF(word_order=2)  # chrF++
ter = TER()


def corpus_scores(df_model: pd.DataFrame) -> dict:
    preds = df_model["translated_en"].fillna("").astype(str).tolist()
    refs = [df_model["reference_en"].fillna("").astype(str).tolist()]
    return {
        "BLEU": bleu.corpus_score(preds, refs).score,
        "chrF++": chrf.corpus_score(preds, refs).score,
        "TER": ter.corpus_score(preds, refs).score,
    }


def add_length_bins(df_model: pd.DataFrame) -> pd.DataFrame:
    out = df_model.copy()
    out["src_len_words"] = out["source_vi"].astype(str).str.split().str.len()
    out["length_bin"] = pd.cut(
        out["src_len_words"],
        bins=[0, 10, 20, 35, 10000],
        labels=["1-10", "11-20", "21-35", "36+"],
        include_lowest=True,
    )
    return out


print("Scoring helpers ready")

Scoring helpers ready


In [11]:
gpt_scores = corpus_scores(gpt_eval)
gemini_scores = corpus_scores(gemini_eval)

overall = pd.DataFrame([
    {"Model": "GPT-5.2", **{k: round(v, 3) for k, v in gpt_scores.items()}},
    {"Model": "Gemini-2.5-Flash", **{k: round(v, 3) for k, v in gemini_scores.items()}},
])

overall.to_csv(RESULT_OVERALL, index=False, encoding="utf-8")
print(f"Saved -> {RESULT_OVERALL}")
overall

Saved -> outputs_medev\results_overall.csv


,Model,BLEU,chrF++,TER
0,GPT-5.2,33.383,59.609,59.983
1,Gemini-2.5-Flash,38.274,61.808,54.977


In [12]:
def by_bin_scores(df_model: pd.DataFrame, model_name: str) -> pd.DataFrame:
    dfb = add_length_bins(df_model)
    rows = []
    for b, sub in dfb.groupby("length_bin", observed=True):
        if len(sub) == 0:
            continue
        s = corpus_scores(sub)
        rows.append(
            {
                "Model": model_name,
                "LengthBin": str(b),
                "n": len(sub),
                "BLEU": round(s["BLEU"], 3),
                "chrF++": round(s["chrF++"], 3),
                "TER": round(s["TER"], 3),
            }
        )
    return pd.DataFrame(rows)


by_bin = pd.concat(
    [
        by_bin_scores(gpt_eval, "GPT-5.2"),
        by_bin_scores(gemini_eval, "Gemini-2.5-Flash"),
    ],
    ignore_index=True,
)

by_bin.to_csv(RESULT_BY_BIN, index=False, encoding="utf-8")
print(f"Saved -> {RESULT_BY_BIN}")
by_bin

Saved -> outputs_medev\results_by_length_bin.csv


,Model,LengthBin,n,BLEU,chrF++,TER
0,GPT-5.2,1-10,599,33.936,55.813,63.562
1,GPT-5.2,11-20,2391,31.886,57.882,61.841
2,GPT-5.2,21-35,3378,32.782,59.234,59.810
3,GPT-5.2,36+,2591,34.182,60.532,59.399
4,Gemini-2.5-Flash,1-10,599,37.635,57.476,58.372
5,Gemini-2.5-Flash,11-20,2391,36.024,59.057,56.555
6,Gemini-2.5-Flash,21-35,3378,37.108,60.767,55.196
7,Gemini-2.5-Flash,36+,2591,39.603,63.505,54.211


In [13]:
# Fast bootstrap significance test using sentence-level BLEU deltas (paired)

sent_bleu = BLEU(effective_order=True)


def sentence_bleu(pred: str, ref: str) -> float:
    return sent_bleu.sentence_score(str(pred), [str(ref)]).score


# Align by id on rows where both models are successful
paired = gpt_eval[["id", "reference_en", "translated_en"]].rename(columns={"translated_en": "gpt_en"}).merge(
    gemini_eval[["id", "translated_en"]].rename(columns={"translated_en": "gemini_en"}),
    on="id",
    how="inner",
)

print(f"Paired successful rows: {len(paired):,}")

paired["gpt_sbleu"] = paired.apply(lambda r: sentence_bleu(r["gpt_en"], r["reference_en"]), axis=1)
paired["gemini_sbleu"] = paired.apply(lambda r: sentence_bleu(r["gemini_en"], r["reference_en"]), axis=1)
deltas = (paired["gemini_sbleu"] - paired["gpt_sbleu"]).to_numpy()


def bootstrap_mean_delta(values: np.ndarray, n_boot: int = 120, seed: int = 42):
    rng = np.random.default_rng(seed)
    n = len(values)
    means = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        means.append(values[idx].mean())
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(np.mean(means)), float(lo), float(hi)


mean_diff, ci_lo, ci_hi = bootstrap_mean_delta(deltas, n_boot=N_BOOT, seed=RANDOM_SEED)
print(f"Sentence-BLEU diff (Gemini - GPT): {mean_diff:.3f}")
print(f"95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]")
print("Significant" if ci_lo > 0 or ci_hi < 0 else "Not significant")

Paired successful rows: 8,959
Sentence-BLEU diff (Gemini - GPT): 3.730
95% CI: [3.430, 4.035]
Significant
